# Datos faltantes


Usa esta plantilla cuando haya muchos `NaN` o el enunciado pregunte cómo tratarlos.

La decisión importante: un valor faltante puede ser ruido, pero también puede tener significado. Por ejemplo, que una prueba médica no esté medida puede decir algo del paciente.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "target"  # CAMBIAR si hay target. Si no hay, adapta el split.

df = pd.read_csv("data.csv")

missing = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_report = pd.DataFrame({"missing": missing, "percent": missing_percent})
print(missing_report[missing_report["missing"] > 0])


## Regla práctica

- Pocos NaN en numéricas: mediana.
- Pocos NaN en categóricas: moda.
- Muchos NaN en una columna: pensar si eliminarla o crear bandera.
- NaN con significado: crear columna `columna_was_missing`.


In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

# Ejemplo: crear banderas para columnas con muchos faltantes.
# Hazlo antes del pipeline para train y test por igual.
cols_with_missing_flags = [col for col in X_train.columns if X_train[col].isna().mean() > 0.2]

for col in cols_with_missing_flags:
    X_train[f"{col}_was_missing"] = X_train[col].isna().astype(int)
    X_test[f"{col}_was_missing"] = X_test[col].isna().astype(int)

numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ]
)

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Train procesado:", X_train_ready.shape)
print("Test procesado:", X_test_ready.shape)


## Cómo explicarlo

No digas solo “he rellenado los NaN”. Mejor:

“Imputo numéricas con mediana porque es robusta a outliers; categóricas con moda porque mantiene una categoría válida; y para columnas con muchos faltantes añado una bandera porque la ausencia puede ser informativa.”


## Si falla

- Error por NaN después del pipeline: hay columnas no incluidas o target con NaN.
- Demasiadas columnas tras one-hot: hay categóricas con muchos valores distintos.
- Resultado malo: quizá faltaba una bandera de ausencia.
- Si una columna tiene 95% NaN, probablemente no conviene tratarla como una variable normal.
